In [1]:
import json
import os
from typing import List
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, messages_from_dict, messages_to_dict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

# =====================================================================

class JSONChatMessageHistory(BaseChatMessageHistory):
    """Кастомный класс истории сообщений, сохраняющий данные в JSON файл."""
    
    def __init__(self, session_id: str, file_directory: str = "chat_histories"):
        self.session_id = session_id
        self.file_directory = file_directory
        # Создаем папку для историй, если её нет
        os.makedirs(self.file_directory, exist_ok=True)
        self.file_path = os.path.join(self.file_directory, f"{session_id}.json")

    @property
    def messages(self) -> List[BaseMessage]:
        """Загружает сообщения из JSON-файла при каждом обращении."""
        if not os.path.exists(self.file_path):
            return []
        try:
            with open(self.file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                # Восстанавливаем объекты сообщений LangChain из словаря
                return messages_from_dict(data)
        except (json.JSONDecodeError, KeyError):
            return []

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """Добавляет новые сообщения в JSON-файл."""
        # 1. Читаем текущую историю
        current_messages = self.messages
        # 2. Добавляем новые сообщения
        current_messages.extend(messages)
        # 3. Сериализуем объекты сообщений в стандартный формат словарей LangChain
        serializable_data = messages_to_dict(current_messages)
        # 4. Перезаписываем файл
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(serializable_data, f, ensure_ascii=False, indent=2)

    def clear(self) -> None:
        """Очищает историю (удаляет файл)."""
        if os.path.exists(self.file_path):
            os.remove(self.file_path)

# =====================================================================

llm = ChatOpenAI(
    model="maple",
    base_url="http://localhost:8080/v1", 
    api_key="not-needed",
    temperature=0.6,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты ИИ-ассистент. Отвечай лаконично."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

base_chain = prompt | llm

def get_json_history(session_id: str) -> JSONChatMessageHistory:
    """Функция-фабрика для RunnableWithMessageHistory."""
    return JSONChatMessageHistory(session_id=session_id)

# Оборачиваем цепочку для автоматической работы с нашей JSON-памятью
bot_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history=get_json_history,
    input_messages_key="user_input",
    history_messages_key="chat_history"
)
#
def chat(bot_chain):
    print("История сохраняется в папку 'chat_histories'.")
    print("Для выхода введите 'пока' 'выход', 'exit' или 'q'\n")

    SESSION_ID = input("Введите идентификатор\n>").strip()

    config = {"configurable": {"session_id": SESSION_ID}}

    print(f"Pei Kib Breat слушает...(Голос: {SESSION_ID})")

    while True:
        try:
            user_message = input("> ").strip()
            print(user_message)
            
            if user_message.lower() in ['пока', 'выход', 'exit', 'q']:
                print("Pei Kib Breat: Пока!")
                break
                
            if not user_message:
                continue
                
            # Отправка сообщения
            response = bot_chain.invoke(
                {"user_input": user_message}, 
                config=config
            )
            
            print(f"Pei Kib Breat: {response.content}\n")
            
        except KeyboardInterrupt:
            print("\nPei Kib Breat: Сессия завершена. Пока!")
            break

/home/a/anaconda3/envs/Statistic/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [2]:
chat(bot_chain) # Первый пользователь

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 1


Pei Kib Breat слушает...(Голос: 1)


>  Привет, меня зовут Семён


Привет, меня зовут Семён
Pei Kib Breat: Привет, Семён! Чем могу помочь?



...

Я биоинформатик и занимаюсь анализом транскриптомных данных, ML и статистикой
Pei Kib Breat: Отлично, Семён. Звучит как классный набор задач для биоинформатики. Работа с транскриптомными данными (RNA-seq, microarrays, single-cell RNA-seq) часто требует комбинации временных рядов, машинного обучения и статистики. 

Чем могу быть полезен? Например, по анализу временных экспрессий, построению моделей регуляции, статистической генетике, или подборе алгоритмов ML для биологических данных.



>  Я умею писать на Python, R и bash


Я умею писать на Python, R и bash
Pei Kib Breat: Отлично, Python, R и bash — это тройка для биоинформатики. Python популярен для ML и библиотек вроде pandas, scikit-learn, biopython. R — для статистики и визуализации (ggplot2, limma). Bash — для автоматизации и обработки файлов на Linux. 

Можешь ли ты участвовать в проектах, где комбинируются эти языки? Например, пайплайн на Python/R для постановки моделей и их валидации в bash-скриптах.



>  пока


пока
Pei Kib Breat: Пока!


In [3]:
chat(bot_chain)# Второй пользователь

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 2


Pei Kib Breat слушает...(Голос: 2)


>  Привет, я Емельян


Привет, я Емельян
Pei Kib Breat: Привет, Емельян! Чем могу помочь?



...

Я врач-исследователь, занимаюсь изучением клинической биохимии
Pei Kib Breat: Привет, Емельян! Рад познакомиться. Если есть вопросы по клинической биохимии или другие темы — обращайся.



>  Я умею ставить реакции ИФА, знаю спектрометрию и могу ставить ПЦР-реакции


Я умею ставить реакции ИФА, знаю спектрометрию и могу ставить ПЦР-реакции
Pei Kib Breat: Отлично, Емельян. Знаком с методикой: ИФА, спектрометрия, ПЦР. Если есть вопросы по анализу, интерпретации результатов, сравнению методов или клиническим примерам — обращайся. Могу помочь с биохимическими проективами, качеством испытаний или методологией.



>  пока


пока
Pei Kib Breat: Пока!


In [5]:
chat(bot_chain) # Первый пользователь reuse

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 1


Pei Kib Breat слушает...(Голос: 1)


>  Привет, помнишь меня?


Привет, помнишь меня?
Pei Kib Breat: Привет, Семён! Да, помню тебя. Рад снова. Ты занимаешься биоинформатикой, пишешь на Python/R/bash — звучит как умный коктейль из методов анализа данных, машинного обучения и системного администрирования. 

Готов помочь с кодом, анализом данных, объяснением концепций или идеями для проектов. Что сейчас в планах?



>  Какими методами анализа данных я владею?


Какими методами анализа данных я владею?
Pei Kib Breat: Отличный вопрос. В рамках биоинформатики и моего «опыта» я владею следующим:

**1. Классическая статистика и биостатистика:**  
- Регрессионный анализ (линейный, логистический), ANOVA  
- Манипулятивные дизайны (CRISPR-эксперименты)  
- Статистическое моделирование (модели времени, выживаемость)  
- Корреляционный и причинно-следственный анализ (например, через динамические модели)

**2. Машинное обучение и глубокое обучение:**  
- Пайплайны признаков, отбор признаков (RFECV, LASSO)  
- Статистические модели ML (RANDOM FOREST, XGBoost, SVM)  
- Глубокое обучение для транскрипто-геномики (кластеризация генов, предсказание регуляции)  
- Генеративные модели (GAN, VAE) для синтеза данных

**3. Специализированные биоинформатические методы:**  
- Анализ временных рядов (импульс-корреляция, ARIMA, нейросетевые модели для экспрессии Genes)  
- Методы работы с многокритериальным анализом (множественное тестирование генов)  
- Статистика в

>  пока


пока
Pei Kib Breat: Пока!


In [6]:
chat(bot_chain) # Второй пользователь reuse

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 2


Pei Kib Breat слушает...(Голос: 2)


>  Привет, ты запомнил, чем я занимаюсь?


Привет, ты запомнил, чем я занимаюсь?
Pei Kib Breat: Да, запомнил, Емельян. Вы исследуете клиническую биохимию методами: ИФА, спектрометрией и ПЦР-реакциями. Готов помочь с анализом результатов, объяснением методик, сравнением анализаторов или практическими вопросами в вашей области.



>  пока


пока
Pei Kib Breat: Пока!


___